## MPFIT Light Curve Fitting — DESIRT Data

This notebook uses an FPCA model to fit DESIRT light curves from a local directory, saving intermediate and final results to CSV files. It first creates raw.csv with one row per light curve and filter, then reshapes this into wide.csv with one row per light curve combining all filters, and finally converts apparent magnitudes to absolute magnitudes in des_final_data.csv.

#### Inputs
- `eigen_val.txt`: FPCA eigenvectors
- Light curve CSVs with columns: `mjd`, `mag`, `magerr`, `filter`, `redshift`, `name`

#### Outputs
- desirt_final_data.csv: FPCA parameters
- plots/: light curve plots


## 1. Imports

In [52]:
from astropy.table import Table
import astropy.units as u
import numpy as np
import os
import pandas as pd
from matplotlib import pyplot as plt
from scipy.interpolate import interp1d
import warnings
from math import sqrt
from numpy import square
from mpfit import mpfit
import random
import math
import pandas as pd
import numpy as np
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u

warnings.filterwarnings("ignore")

## 2. Configuration 


In [53]:
num_attempts=3 ##No. of times mpfit is run per loghtcurve per filter to determine the bets fit
min_no_points=5  ##Minimum number of points required in a particular filter 


##Filter Configurations

color=['green', 'red',  'blue']
dict_color={ 'g':color[0], 'r':color[1], 'z':color[2]}
dict_number={ 'g':3.0, 'r':1.0,  'z':-1.0}
dict_sym={'g':'s', 'r':'*',  'z':'^'}
dict_legend={ 'g':'g+3.0', 'r':'r+1.0', 'z':'z-1.0'}
dict_ms={'g':40, 'r':75, 'z':60}

##Directories

root_dir = "/Volumes/new_drive/Desirt_Work" ##Directory where the folders containing the  csv files (light curves) are located
os.makedirs('tables', exist_ok=True) ##Directory to save tables
os.makedirs('plots', exist_ok=True)  ##Directory to save figures

##Output_Files
raw_file="raw.csv"
wide_file="wide.csv"
output_file="desirt_final_data.csv"

## 3. Load Eigenvectors & Mapping Coefficients


In [54]:
mapping_coeff=pd.read_csv('mapping_coeff_original.csv')
mapping_coeff = mapping_coeff.set_index('Filter')


eigen_val=np.genfromtxt('eigen_val.txt')
ph=eigen_val[1:,0]
vec0=eigen_val[1:,1]
vec1=eigen_val[1:,2]
vec2=eigen_val[1:,3]
imod0 = interp1d(ph, vec0, fill_value='extrapolate')
imod1 = interp1d(ph, vec1, fill_value='extrapolate')
imod2 = interp1d(ph, vec2, fill_value='extrapolate')


## 4. Define Functions


In [55]:
def leng_calc1(time_array2, t_o2):  
    mphase = (time_array2-t_o2)/(1+redshift_p)
    indsel=(np.where((mphase>=-10.0) & (mphase<=40.0)))[0]
    return len(indsel)


def modelcurve1(x_calc, theta, arr_pass):
    t_min, mag_o, a1, a2 = theta
    mphase = (x_calc-t_min)/(1+redshift_p)

    if mphase >= -10.0 and mphase <= 40.0:
        VecP0 = float(imod0(mphase))
        VecP1 = float(imod1(mphase))
        VecP2 = float(imod2(mphase))

        y_calc = mag_o + VecP0 + a1*VecP1 + a2*VecP2
       
        err_pc_scores=np.square(arr_pass[0]* VecP1)+np.square(arr_pass[1]*VecP2)
      
        
        err_cal= err_pc_scores
      
    else:
        y_calc=np.nan
        err_cal=np.nan
          
    return y_calc, err_cal



def myfunctlin1(p, fjac=None, x=None, y=None, err=None, arr_pass=None, ini_params=None, priors=None):
    
    dev = np.zeros((len(x)), dtype = np.float64)
    pr_arr = np.zeros((2), dtype = np.float64)
   
    
    for kk in range(len(dev)):
        
        y_calc, tem=modelcurve1(x[kk], p, arr_pass)
        
        if not np.isnan(y_calc):
            dev[kk]=(y[kk]-y_calc)/sqrt(np.square(err[kk])+ tem)
    
    pr_arr[0]=(p[2]-priors[0])/(sqrt(2)*arr_pass[0])
    pr_arr[1]=(p[3]-priors[1])/(sqrt(2)*arr_pass[1])


    dev_f=np.concatenate((dev, pr_arr))   
    status = 0
    return [status, dev_f]


def mpfit_run_flux(time_array, data_vec, data_vec_unc, ini_params, tym_cons_low, tym_cons_high, arr_pass, priors, redshift_p):

    leng=leng_calc1(time_array, ini_params[0]) 
    if(leng<min_no_points):
        
        return np.zeros((4)), np.zeros((4)), 10000, 1
    p0=ini_params
   
    priors=priors
    parbase={'value':0., 'fixed':0, 'limited':[0,0], 'limits':[0.,0.]}
    parinfo=[]
    for i in range(len(p0)):
        parinfo.append({'value':0., 'fixed':0, 'limited':[0,0], 'limits':[0.,0.]})
    for i in range(len(p0)): 
        parinfo[i]['value']=p0[i]

    parinfo[0]['limited'][0] = 1
    parinfo[0]['limits'][0]  = tym_cons_low
    parinfo[0]['limited'][1] = 1
    parinfo[0]['limits'][1]  = tym_cons_high


    parinfo[2]['limited'][0] = 1
    parinfo[2]['limits'][0]  = -10
    parinfo[2]['limited'][1] = 1
    parinfo[2]['limits'][1]  = 10

    parinfo[3]['limited'][0] = 1
    parinfo[3]['limits'][0]  = -10
    parinfo[3]['limited'][1] = 1
    parinfo[3]['limits'][1]  = 10


    fa = {'x':time_array, 'y':data_vec, 'err':data_vec_unc, 'arr_pass': arr_pass, 'ini_params':ini_params, 'priors': priors}
    
    m = mpfit(myfunctlin1,  parinfo=parinfo, quiet=True, functkw=fa)

    if (m.status <= 0): 
        #print( 'error message = ', m.errmsg)
        am=0.0
    else:
        am=m.fnorm

    #print(m.params, m.perror, am)
    return m.params, m.perror, am, leng-2


## 5. Main loop over light curves

In [ ]:
##b1_now, b2_now are the priors in the 2 FPCA scores

data_full=[]
table_full=[]

for file in os.listdir(root_dir):
        path1=os.path.join(root_dir, file)
        if(os.path.isdir(path1)):

            # if(path1=="/Volumes/new_drive/Desirt_Work/Desirt_Work_Test"):
            #     continue
            # if(path1=="/Volumes/new_drive/Desirt_Work/Mapping"):
            #     continue
                
            for file1 in os.listdir(path1):
                
                path2=os.path.join(path1, file1)
            
                if(path2.endswith(".csv")):
                 
                    data_temp=[]
                    table_temp=[]
                    
                    lightcurve_data=pd.read_csv(path2)
                    redshift_p=lightcurve_data['redshift'][0]
                    
                    name=lightcurve_data['name'][0]
                      
                    lightcurve_data=lightcurve_data.loc[lightcurve_data['magerr'] <0.5]
                    mjd = lightcurve_data['mjd'].to_numpy()
                    mag = lightcurve_data['mag'].to_numpy()
                    magerr = lightcurve_data['magerr'].to_numpy()
                    filter= lightcurve_data['filter'].to_numpy()
                    

                    count_filter=0
                    array_mag=[]
                    array_time=[]
                    array_err=[]
                    fil=[]

                    g=[]
                    r=[]
                    z=[]
                  
                    gerr=[]
                    rerr=[]
                    zerr=[]
                    
                    g_t=[]
                    r_t=[]
                    z_t=[]
                    
    
    
                    for m in range(len(filter)):
                        if(filter[m]=='g'):
                            g.append(mag[m])
                            gerr.append(magerr[m])
                            g_t.append(mjd[m])
                            

                        elif(filter[m]=='r'):
                            r.append(mag[m])
                            rerr.append(magerr[m])
                            r_t.append(mjd[m])
                            


                        elif(filter[m]=='z'):
                            z.append(mag[m])
                            zerr.append(magerr[m])
                            z_t.append(mjd[m])
                            
                    
            
                    if(g):
                        g=np.array(g)
                        if(len(g)>min_no_points-1):
                            count_filter=count_filter+1
                            gerr=np.array(gerr)
                            g_t=np.array(g_t)
                            array_mag.append(g)
                            array_time.append(g_t)
                            array_err.append(gerr)
                            fil.append('g')
                          
                    
                    if(r):
                        r=np.array(r)
                        if(len(r)>min_no_points-1):
                            count_filter=count_filter+1
                            rerr=np.array(rerr)
                            r_t=np.array(r_t)
                            array_mag.append(r)
                            array_time.append(r_t)
                            array_err.append(rerr)
                            fil.append('r')



                    if(z):
                        z=np.array(z)
                        if(len(z)>min_no_points-1):
                            count_filter=count_filter+1
                            zerr=np.array(zerr)
                            z_t=np.array(z_t)
                            array_mag.append(z)
                            array_time.append(z_t)
                            array_err.append(zerr)
                            fil.append('z')
                          
 

                    if(count_filter>0):

                        a_arr=np.zeros((len(array_mag)))
                        for kkk in range(len(array_mag)):
                            a_arr[kkk]=len(array_mag[kkk])
                        b_arr=np.argsort(-a_arr)

                        fil = [fil[ipip] for ipip in b_arr]
                        array_mag = [array_mag[ipip] for ipip in b_arr]
                        array_time = [array_time[ipip] for ipip in b_arr]
                        array_err = [array_err[ipip] for ipip in b_arr]

    
                        for fil_index in range(count_filter):

                            if(fil[fil_index]=='g'):
                                
                                b1_now=redshift_p*redshift_p*mapping_coeff.loc['g', 'm1']+mapping_coeff.loc['g', 'c1']*redshift_p+mapping_coeff.loc['g', 'c0']
                                b2_now=redshift_p*mapping_coeff.loc['g', 'm2']+mapping_coeff.loc['g', 'c2']
                                    
                                sigma_b1_now=math.pow(mapping_coeff.loc['g', 'std_m1'],2)*math.pow(redshift_p,4)+math.pow(mapping_coeff.loc['g', 'std_c1'],2)*math.pow(redshift_p,2)+math.pow(mapping_coeff.loc['g', 'std_c0'],2)
                                sigma_b2_now=square(mapping_coeff.loc['g', 'std_m2']*redshift_p)+square(mapping_coeff.loc['g', 'std_c2'])
                                

                            elif(fil[fil_index]=='r'):
                                b1_now=redshift_p*redshift_p*mapping_coeff.loc['r', 'm1']+mapping_coeff.loc['r', 'c1']*redshift_p+mapping_coeff.loc['r', 'c0']
                                b2_now=redshift_p*mapping_coeff.loc['r', 'm2']+mapping_coeff.loc['r', 'c2']

                                sigma_b1_now=math.pow(mapping_coeff.loc['r', 'std_m1'],2)*math.pow(redshift_p,4)+math.pow(mapping_coeff.loc['r', 'std_c1'],2)*math.pow(redshift_p,2)+math.pow(mapping_coeff.loc['r', 'std_c0'],2)
                                sigma_b2_now=square(mapping_coeff.loc['r', 'std_m2']*redshift_p)+square(mapping_coeff.loc['r', 'std_c2'])


                            elif(fil[fil_index]=='z'):
                                b1_now=redshift_p*mapping_coeff.loc['z', 'm1']+mapping_coeff.loc['z', 'c1']
                                b2_now=redshift_p*mapping_coeff.loc['z', 'm2']+mapping_coeff.loc['z', 'c2']
                                
                                sigma_b1_now=square(mapping_coeff.loc['z', 'std_m1']*redshift_p)+square(mapping_coeff.loc['z', 'std_c1'])
                                sigma_b2_now=square(mapping_coeff.loc['z', 'std_m2']*redshift_p)+square(mapping_coeff.loc['z', 'std_c2'])
                            
                            
                            arr_pass=np.array([sigma_b1_now, sigma_b2_now])
                            priors=[b1_now, b2_now]
                    
                            if(fil_index==0):
                                chi1=np.zeros(num_attempts)
                                final_params1=np.zeros((num_attempts,4))
                                final_errors1=np.zeros((num_attempts,4))
                                leng1=np.zeros(num_attempts)

                                final_params=np.zeros(4)
                                final_errors=np.zeros(4)

                                t_cons1=(array_time[0])[np.argmin(array_mag[0])]
                                

                                for mmmm in range(num_attempts):
                                
                                    ini_params=np.array([t_cons1+random.uniform(-3,3)*(1+redshift_p), np.min(array_mag[fil_index]), b1_now+random.uniform(-2,2), b2_now+random.uniform(-2,2)])

                                    final_params1[mmmm,:], final_errors1[mmmm,:], chi1[mmmm], leng1[mmmm] = mpfit_run_flux(array_time[fil_index], array_mag[fil_index], array_err[fil_index], ini_params, t_cons1-10*(1+redshift_p), t_cons1+10*(1+redshift_p), arr_pass, priors, redshift_p)
                                

                                min_ind=np.argmin(chi1)
                                final_params[:]=final_params1[min_ind,:]
                                final_errors[:]=final_errors1[min_ind,:]
                                chi=chi1[min_ind]
                                leng=leng1[min_ind]

                        
                                if(chi<10000):

                                    t_cons2=final_params[0]
                                    
                                    data_temp.append([final_params[0], final_params[1], final_params[2], final_params[3],
                                                    final_errors[0], final_errors[1], final_errors[2], final_errors[3], 
                                                    chi, fil[fil_index], name, leng, redshift_p])
                                    
                                    data_full.append([final_params[0], final_params[1], final_params[2], final_params[3],
                                                    final_errors[0], final_errors[1], final_errors[2], final_errors[3], 
                                                    chi, fil[fil_index], name, leng,  redshift_p])
                    
                                    tmp_table = Table([array_time[fil_index], array_mag[fil_index], array_err[fil_index]],
                                                    names=('time', 'mag', 'magerr'),
                                                    meta={'redshift': redshift_p, 'filter':fil[fil_index], 'name':name, 'path':file1})
                                        
                                    tmp_table.write("tables/"+name+'_'+fil[fil_index]+'.csv', overwrite=True)    
                                    table_temp.append(tmp_table)
                                    table_full.append(tmp_table)

                                else:
                                    break


                            else:
                                
                                chi1=np.zeros(num_attempts)
                                final_params1=np.zeros((num_attempts,4))
                                final_errors1=np.zeros((num_attempts,4))
                                leng1=np.zeros(num_attempts)

                                final_params=np.zeros(4)
                                final_errors=np.zeros(4)

                             

                                for mmmm in range(num_attempts):
                                
                                    ini_params=np.array([t_cons2+random.uniform(-2,2)*(1+redshift_p), np.min(array_mag[fil_index]), b1_now+random.uniform(-2,2), b2_now+random.uniform(-2,2)])
                                    final_params1[mmmm,:], final_errors1[mmmm,:], chi1[mmmm], leng1[mmmm] = mpfit_run_flux(array_time[fil_index], array_mag[fil_index], array_err[fil_index], ini_params, t_cons2-7*(1+redshift_p), t_cons2+7*(1+redshift_p), arr_pass, priors, redshift_p)
                                    
                                min_ind=np.argmin(chi1)
                                final_params[:]=final_params1[min_ind,:]
                                final_errors[:]=final_errors1[min_ind,:]
                                chi=chi1[min_ind]
                                leng=leng1[min_ind]

                        
                                if(chi<10000):
                                    
                                    data_temp.append([final_params[0], final_params[1], final_params[2], final_params[3],
                                                    final_errors[0], final_errors[1], final_errors[2], final_errors[3], 
                                                    chi, fil[fil_index], name, leng, redshift_p])
                                    
                                    data_full.append([final_params[0], final_params[1], final_params[2], final_params[3],
                                                    final_errors[0], final_errors[1], final_errors[2], final_errors[3], 
                                                    chi, fil[fil_index], name, leng, redshift_p])
                                    
                                    tmp_table = Table([array_time[fil_index], array_mag[fil_index], array_err[fil_index]],
                                                    names=('time', 'mag', 'magerr'),
                                                    meta={'redshift': redshift_p, 'filter':fil[fil_index], 'name':name, 'path':file1})
                                    
                                    tmp_table.write("tables/"+name+'_'+fil[fil_index]+'.csv', overwrite=True)
                                    table_temp.append(tmp_table)
                                    table_full.append(tmp_table)

                        

                        data_temp=np.array(data_temp)
                        df = pd.DataFrame(data_temp)
                        df.to_csv("mpfit_temp.csv")
                        model=pd.read_csv("mpfit_temp.csv")
                        model=np.array(model)
                        model=np.delete(model, 0, axis=1)
                        legen=[]


                        if(len(model)>0):

                            for j in range(len(model)):
                                #table_temp[j].write('tables/'+str(model[j,-1])+'_'+(model[j,9])[-1]+'.csv', format='csv', overwrite=True)

                                plo_data_x=eigen_val[:,0]*(1+model[j,-1])+model[j,0]-model[0,0]
                                plo_data_y=model[j,1] + eigen_val[:,1] + model[j,2]*eigen_val[:,2] + model[j,3]*eigen_val[:,3]+dict_number[model[j,9]]
                            
                                plt.plot(plo_data_x, plo_data_y, color=dict_color[model[j,9]],   linestyle='solid', label='_nolegend_') 
                                
                                plt.errorbar(table_temp[j][0][:] - model[0, 0], table_temp[j][1][:] + dict_number[model[j, 9]],  yerr=table_temp[j][2][:],  fmt=dict_sym[model[j, 9]],   color=dict_color[model[j, 9]],  markersize=int(dict_ms[model[j,9]]/10))

                            
                                coor=250
                                plt.annotate(dict_legend[model[j,9]], xy = (plo_data_x[coor], plo_data_y[coor]), xytext = (1*plo_data_x[coor], 0.99*plo_data_y[coor]), color  = dict_color[model[j,9]])
                
                        
                    
                            plt.xlim([-10*(1+model[0,-1])-5, +40*(1+model[0,-1])+5])
                        
                            plt.xlabel('Time (Days)')
                            plt.ylabel('Magnitude')
                            plt.gca().invert_yaxis()

                           
                            chi=np.round(sum(model[:,8]),2)
                            dof=np.round(sum(model[:,11]),2)
                            plt.title('ID= '+model[0,10]+'  '   + '  '      +'z= '+ str(table_temp[0].meta['redshift']) +'  '  + '  '+    'chisq='+str(chi) + '    dof='+str(dof))
                          
                            plt.savefig("plots/"+model[0, 10]+'.png')
                            plt.close()
        

                                
data_full=np.array(data_full)
df = pd.DataFrame(data_full)
df.to_csv(raw_file, header=['pk_time', 'pk_mag', 'a1', 'a2', 'std_time', 'std_mag',
                                     'std_a1', 'std_a2','chi_2', 'filter', 'name', 'dof', 'redshift' ])
print(f"Length of raw.csv: {len(data_full)}")

print('done')

Length of raw.csv: 281
done


## 6. Wide dataframe



In [57]:

df_long = pd.read_csv(raw_file)
df_long['name'] = df_long['name'].astype(str)
df_long['filter'] = df_long['filter'].astype(str)



# 8 columns per filter to be spread out
value_cols = [
    'pk_time', 'pk_mag', 'a1', 'a2',
    'std_time', 'std_mag', 'std_a1', 'std_a2',
]

# Columns to keep as object metadata (should be consistent across all filters for an object)
meta_cols = [ 'redshift']



# Pivot on 'name' to get one row per object, using 'filter' to spread the columns
pivoted_df = df_long.pivot(
    index='name',
    columns='filter',
    values=value_cols
)


# Flatten the column names to the format 'filter_parameter' (e.g., 'g_alpha')

pivoted_df.columns = [f'{col[1]}_{col[0]}' for col in pivoted_df.columns]
pivoted_df = pivoted_df.reset_index() # Move 'meta_id' from index back to a column



# Extract unique metadata rows and merge them back with the pivoted data
name_df = df_long[['name'] + meta_cols].drop_duplicates(subset=['name']).reset_index(drop=True)

# Merge the metadata onto the wide-format results
final_wide_df = name_df.merge(pivoted_df, on='name', how='right')


final_wide_df.to_csv(wide_file, index=False)

print(f"Final DataFrame has {len(final_wide_df)} rows (one per object).")
print(len(final_wide_df))

Final DataFrame has 113 rows (one per object).
113


## 7. Absolute Magnitudes

In [58]:
# Cosmological Model Setup ---
# Define the Flat Lambda-CDM model
# H0 = 70 km/s/Mpc, Om0 = 0.3
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
print("Cosmological model (Flat Lambda-CDM): H0=70, Om0=0.3")


df=pd.read_csv(wide_file)

MAG_COLUMNS = ['g_pk_mag', 'r_pk_mag', 'z_pk_mag']
REDSHIFT_COLUMN = 'redshift'


# Extract data arrays
apparent_mags = df[MAG_COLUMNS].to_numpy()
redshifts = df[REDSHIFT_COLUMN].to_numpy()
NUM_MAG_BANDS = apparent_mags.shape[1]

# Initialize the output array for absolute magnitudes
absolute_mags = np.zeros_like(apparent_mags, dtype=float)



# Loop through each object (row)
for i in range(len(df)):
    z = redshifts[i]

    # Handle non-physical or zero redshifts
    if z <= 0 or not np.isfinite(z):
        z=0.028
        
    
    # Calculate luminosity distance (d)
    # The result 'ld' is in astropy units (Mpc)
    ld = cosmo.luminosity_distance(z)

    # Convert to parsecs (pc) and get the value (d is required in pc for the formula)
    d_pc = ld.to(u.pc).value


  
    distance_modulus = 5.0 - 5.0 * np.log10(d_pc)

    # Apply the calculation to all magnitude bands for this object
    absolute_mags[i, :] = apparent_mags[i, :] + distance_modulus



# --- 4. Data Replacement and Saving ---
# Create new column names for absolute magnitudes
NEW_MAG_COLUMNS = MAG_COLUMNS # e.g., 'M_g', 'M_r'

# Replace the apparent magnitude columns with the calculated absolute magnitude columns

# Drop the original apparent magnitude columns if you want to replace them completely
df = df.drop(columns=MAG_COLUMNS)
df[NEW_MAG_COLUMNS] = absolute_mags
# Save the resulting DataFrame to a new CSV file
df.to_csv(output_file, index=False)


print(f"DataFrame now contains absolute magnitude columns: {NEW_MAG_COLUMNS}")

Cosmological model (Flat Lambda-CDM): H0=70, Om0=0.3
DataFrame now contains absolute magnitude columns: ['g_pk_mag', 'r_pk_mag', 'z_pk_mag']
